In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


c:\CETYS\Inteligencia Computacional\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):

        for document in documents:

            self.documents.append(document)
            embedding = self.embedding_model.encode(document.text)
            self.embeddings.append(embedding)


    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:

        query_embedding = self.embedding_model.encode(query)

        results = []

        for i in range(len(self.documents)):

            similarity = cosine_similarity(
                [query_embedding],
                [self.embeddings[i]]
            )[0][0]

            result = SearchResult(
                similarity,
                self.documents[i]
            )

            results.append(result)

        results.sort(
            key=lambda x: x.score,
            reverse=True
        )

        return results[:top_k]

In [13]:
df = pd.read_csv("../Busqueda Semantica/animal-fun-facts-dataset.csv")

In [15]:
documents = []

In [16]:
for i in range(len(df)):

    metadata = {
        "animal_name": str(df.iloc[i]["animal_name"]),
        "source": str(df.iloc[i]["source"]),
        "media_link": str(df.iloc[i]["media_link"]),
        "wikipedia_link": str(df.iloc[i]["wikipedia_link"])
    }

    document = Document(
        text=str(df.iloc[i]["text"]),
        metadata=metadata
    )

    documents.append(document)

In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2")
vector_store = VectorStore(model)
vector_store.add_documents(documents)

c:\CETYS\Inteligencia Computacional\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrgok\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3646.02it/s]


In [21]:
queries = [
    "aquatic animals",
    "animals flight",
    "dangerous animals",
    "smart animals",
    "fast animals"
]

for query in queries:

    print("Query:", query)

    results = vector_store.search(query, top_k=3)

    for result in results:

        print("\nScore:", result.score)

        print("\nText:")
        print(result.document.text)

        print("\nMetadata:")
        print(result.document.metadata)

        print("\n-----------------------------------")

Query: aquatic animals

Score: 0.6826409

Text:
They are semi-aquatic mammals..
They live in and around lakes, rivers, swamps, and tropical rivers. They are excellent at swimming and can hold their breath for 5 minutes at a time. They are well adapted for this, with partially webbed toes for swimming. Also, their eyes, ears and nose are high on their heads to watch out for predators when they are underwater. 2

Metadata:
{'animal_name': 'capybara', 'source': 'https://factanimal.com/capybara/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Capybara'}

-----------------------------------

Score: 0.6296179

Text:
They swim like frogs

Metadata:
{'animal_name': 'grebe', 'source': 'https://a-z-animals.com/animals/grebe/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Grebe'}

-----------------------------------

Score: 0.6263706

Text:
They are aquatic, and although they posses rudimentary lungs, they breathe primarily through their gills and, to a lesser extent, their skin.

Metadata:
{'an

In [23]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
         
         for document in documents:

            self.documents.append(document)
            embedding = self.embedding_model.encode(document.text)
            self.embeddings.append(embedding)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embedding = self.embedding_model.encode(query)

        results = []

        for i in range(len(self.documents)):

            document = self.documents[i]
            valid_document = True

            if metadata_filter is not None:

                for key in metadata_filter:

                    if document.metadata.get(key) != metadata_filter[key]:
                        valid_document = False

            if valid_document == False:
                continue

            similarity = cosine_similarity(
                [query_embedding],
                [self.embeddings[i]]
            )[0][0]

            result = SearchResult(
                similarity,
                document
            )

            results.append(result)

        results.sort(
            key=lambda x: x.score,
            reverse=True
        )

        return results[:top_k]

In [24]:
news_df = pd.read_csv("../Busqueda Semantica/abcnews-date-text.csv")
news_documents = []

In [25]:
for i in range(5000):

    metadata = {
        "publish_date": str(news_df.iloc[i]["publish_date"])
    }

    document = Document(
        text=str(news_df.iloc[i]["headline_text"]),
        metadata=metadata
    )

    news_documents.append(document)

In [26]:
news_store = FilteredVectorStore(model)
news_store.add_documents(news_documents)

In [27]:
examples = [

    {
        "query": "sports news",
        "filter": {
            "publish_date": "20030219"
        }
    },

    {
        "query": "politics",
        "filter": {
            "publish_date": "20030220"
        }
    },

    {
        "query": "technology",
        "filter": {
            "publish_date": "20030221"
        }
    },

    {
        "query": "crime news",
        "filter": {
            "publish_date": "20030222"
        }
    },

    {
        "query": "economy",
        "filter": {
            "publish_date": "20030223"
        }
    }
]

In [28]:
for example in examples:

    print("QUERY:", example["query"])
    print("FILTER:", example["filter"])

    results = news_store.search(
        query=example["query"],
        top_k=3,
        metadata_filter=example["filter"]
    )

    for result in results:

        print("\nScore:", result.score)

        print("\nHeadline:")
        print(result.document.text)

        print("\nMetadata:")
        print(result.document.metadata)

        print("\n-----------------------------------")

QUERY: sports news
FILTER: {'publish_date': '20030219'}

Score: 0.34826663

Headline:
stop changing the rules fans tell afl

Metadata:
{'publish_date': '20030219'}

-----------------------------------

Score: 0.28410816

Headline:
big hopes for launceston cycling championship

Metadata:
{'publish_date': '20030219'}

-----------------------------------

Score: 0.23933867

Headline:
last minute call hands alinghi big lead

Metadata:
{'publish_date': '20030219'}

-----------------------------------
QUERY: politics
FILTER: {'publish_date': '20030220'}

Score: 0.3629577

Headline:
politicians breeding publics anti troop sentiment

Metadata:
{'publish_date': '20030220'}

-----------------------------------

Score: 0.28375798

Headline:
nsw coalition making magic pudding election

Metadata:
{'publish_date': '20030220'}

-----------------------------------

Score: 0.23347272

Headline:
more anti war rallies planned

Metadata:
{'publish_date': '20030220'}

-----------------------------------
QU

Pude ver como funcionan los vector stores y busquedas semanticas usando embeddings generados, implementarlo manualmente me ayudo a entender mejor como se guardan los documento y como se comparan con la similitud coseno y como se obtienen los mejores resultados sin usar sin usar librerías avanzadas como FAISS o LangChain, las cuales harian la mayoria del trabajo automaticamente.